In [2]:
# Cell 1: Imports and load all raw data we saved in notebook 01
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

# Load saved data
uk_cpi = pd.read_csv('../data/raw/uk_cpi.csv', index_col=0, parse_dates=True)
uk_wages = pd.read_csv('../data/raw/uk_wages.csv', index_col=0, parse_dates=True)
uk_unemp = pd.read_csv('../data/raw/uk_unemployment.csv', index_col=0, parse_dates=True)
oil = pd.read_csv('../data/raw/brent_crude.csv', index_col=0, parse_dates=True)
gbp_usd = pd.read_csv('../data/raw/gbp_usd.csv', index_col=0, parse_dates=True)
us_cpi = pd.read_csv('../data/raw/us_cpi.csv', index_col=0, parse_dates=True)

print("All datasets loaded!")
print(f"CPI shape: {uk_cpi.shape}")
print(f"Wages shape: {uk_wages.shape}")
print(f"Unemployment shape: {uk_unemp.shape}")
print(f"Oil shape: {oil.shape}")
print(f"GBP/USD shape: {gbp_usd.shape}")
print(f"US CPI shape: {us_cpi.shape}")

All datasets loaded!
CPI shape: (341, 1)
Wages shape: (303, 1)
Unemployment shape: (313, 1)
Oil shape: (6916, 1)
GBP/USD shape: (6914, 1)
US CPI shape: (317, 1)


In [3]:
# Cell 2: Convert daily data to monthly averages
# 'MS' means Month Start - so every date will be the 1st of the month
# This makes it easy to merge with our monthly CPI data later

oil_monthly = oil.resample('MS').mean()
gbp_monthly = gbp_usd.resample('MS').mean()

# US CPI is already monthly but let's make sure it's on Month Start too
us_cpi_monthly = us_cpi.resample('MS').mean()

print("Daily → Monthly conversion done!")
print(f"Oil monthly shape: {oil_monthly.shape}")
print(f"GBP/USD monthly shape: {gbp_monthly.shape}")
print(f"\nLatest oil monthly average: {oil_monthly.iloc[-1].values[0]:.2f} USD/barrel")
print(f"Latest GBP/USD monthly average: {gbp_monthly.iloc[-1].values[0]:.4f}")

Daily → Monthly conversion done!
Oil monthly shape: (319, 1)
GBP/USD monthly shape: (319, 1)

Latest oil monthly average: 69.00 USD/barrel
Latest GBP/USD monthly average: 1.3322


In [4]:
# Drop the current incomplete month from daily-converted series
# since we're only partway through June, the average would be misleading
oil_monthly = oil_monthly.iloc[:-1]
gbp_monthly = gbp_monthly.iloc[:-1]

print("Latest complete month oil:", oil_monthly.index[-1].date(), 
      f"= ${oil_monthly.iloc[-1].values[0]:.2f}/barrel")
print("Latest complete month GBP/USD:", gbp_monthly.index[-1].date(), 
      f"= {gbp_monthly.iloc[-1].values[0]:.4f}")

# Now merge everything into one dataframe
df = uk_cpi.copy()
df = df.join(uk_wages, how='left')
df = df.join(uk_unemp, how='left')
df = df.join(oil_monthly, how='left')
df = df.join(gbp_monthly, how='left')
df = df.join(us_cpi_monthly, how='left')

print(f"\nMerged dataframe shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nDate range: {df.index[0].date()} to {df.index[-1].date()}")
print("\nFirst few rows:")
print(df.head())

Latest complete month oil: 2026-06-01 = $85.40/barrel
Latest complete month GBP/USD: 2026-06-01 = 1.3338

Merged dataframe shape: (341, 6)
Columns: ['cpi', 'wages', 'unemployment', 'oil_price', 'gbp_usd', 'us_cpi']

Date range: 1998-01-01 to 2026-05-01

First few rows:
            cpi  wages  unemployment  oil_price  gbp_usd  us_cpi
date                                                            
1998-01-01  1.5    NaN           NaN        NaN      NaN     NaN
1998-02-01  1.6    NaN           NaN        NaN      NaN     NaN
1998-03-01  1.7    NaN           NaN        NaN      NaN     NaN
1998-04-01  1.8    NaN           NaN        NaN      NaN     NaN
1998-05-01  2.0    NaN           NaN        NaN      NaN     NaN


In [5]:
# Cell 4: Check missing values across the whole dataframe
print("Missing values per column:")
print(df.isnull().sum())

print(f"\nTotal rows: {len(df)}")

# Find the first date where ALL columns have data
first_complete = df.dropna().index[0]
print(f"\nFirst date with complete data across all columns: {first_complete.date()}")

# Trim the dataframe to start from that date
df = df[df.index >= first_complete]

print(f"\nTrimmed dataframe shape: {df.shape}")
print(f"New date range: {df.index[0].date()} to {df.index[-1].date()}")
print("\nMissing values after trimming:")
print(df.isnull().sum())

Missing values per column:
cpi              0
wages           38
unemployment    28
oil_price       24
gbp_usd         24
us_cpi          25
dtype: int64

Total rows: 341

First date with complete data across all columns: 2001-01-01

Trimmed dataframe shape: (305, 6)
New date range: 2001-01-01 to 2026-05-01

Missing values after trimming:
cpi             0
wages           2
unemployment    4
oil_price       0
gbp_usd         0
us_cpi          1
dtype: int64


In [6]:
# Cell 5: Fill remaining missing values (updated syntax for newer pandas)
df = df.ffill()  # forward fill
df = df.bfill()  # backward fill

print("Missing values after filling:")
print(df.isnull().sum())

# Cell 6: Create lag variables
df['oil_lag1'] = df['oil_price'].shift(1)
df['oil_lag2'] = df['oil_price'].shift(2)
df['oil_lag3'] = df['oil_price'].shift(3)
df['gbp_lag1'] = df['gbp_usd'].shift(1)
df['gbp_lag2'] = df['gbp_usd'].shift(2)
df['wages_lag1'] = df['wages'].shift(1)
df['wages_lag3'] = df['wages'].shift(3)

# Drop the first 3 rows which will have NaN from the lagging
df = df.dropna()

print(f"\nFinal dataframe shape: {df.shape}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"\nAll columns: {df.columns.tolist()}")

Missing values after filling:
cpi             0
wages           0
unemployment    0
oil_price       0
gbp_usd         0
us_cpi          0
dtype: int64

Final dataframe shape: (302, 13)
Date range: 2001-04-01 to 2026-05-01

All columns: ['cpi', 'wages', 'unemployment', 'oil_price', 'gbp_usd', 'us_cpi', 'oil_lag1', 'oil_lag2', 'oil_lag3', 'gbp_lag1', 'gbp_lag2', 'wages_lag1', 'wages_lag3']


In [7]:
# Cell 7: Save the clean merged dataset
df.to_csv('../data/processed/merged_clean.csv')

print("Clean dataset saved to data/processed/merged_clean.csv!")
print("\n=== CLEANING COMPLETE ===")
print(f"Observations: {len(df)}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Features: {df.columns.tolist()}")
print("\nFinal snapshot:")
print(df.tail())

Clean dataset saved to data/processed/merged_clean.csv!

=== CLEANING COMPLETE ===
Observations: 302
Date range: 2001-04-01 to 2026-05-01
Features: ['cpi', 'wages', 'unemployment', 'oil_price', 'gbp_usd', 'us_cpi', 'oil_lag1', 'oil_lag2', 'oil_lag3', 'gbp_lag1', 'gbp_lag2', 'wages_lag1', 'wages_lag3']

Final snapshot:
            cpi     wages  unemployment   oil_price   gbp_usd   us_cpi  \
date                                                                     
2026-01-01  3.0  3.676132           4.9   66.602381  1.353565  326.588   
2026-02-01  3.0  3.792774           4.9   70.887000  1.357921  327.460   
2026-03-01  3.3  3.112165           4.9  103.134545  1.333868  330.293   
2026-04-01  2.8  3.112165           4.9  117.287500  1.345405  332.407   
2026-05-01  2.8  3.112165           4.9  107.139474  1.349835  333.979   

              oil_lag1    oil_lag2   oil_lag3  gbp_lag1  gbp_lag2  wages_lag1  \
date                                                                            